# PT-W3-D7 概念实验：LangChat × Ontology × Compiler 集成方案

端到端验证一条语义供应链：世界模型 → 语义治理 → Compiler → Agent 执行，并在运行前应用 Context 与 Policy 约束。

## 实验 1：从业务声明到受控 Agent 计划

用「A101 退租后释放铺位」演示五阶段：Describe、Validate、Assemble、Materialize、Execute。类型在上游定义，事实在底座落地。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager
font_path = '/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc'
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams['font.family'] = font_name
plt.rcParams['axes.unicode_minus'] = False

WORLD_MODEL = {'Space': {'owner': 'Asset Foundation'}, 'Lease': {'owner': 'Lease/Occupancy'}, 'Inspection': {'owner': 'Operations'}}
GOVERNANCE = {'effect_types': {'occupancy'}, 'bindings': {'occupies'}, 'rules': {'release_requires_inspection'}}
DECLARATION = {'goal': 'A101 退租后释放铺位', 'objects': ['Lease', 'Space'], 'binding': 'occupies', 'effect_type': 'occupancy', 'rule': 'release_requires_inspection', 'capability': 'release_space'}
RUNTIME_FACTS = {'lease_id': 'L-001', 'space_id': 'A101', 'lease_state': 'Terminating', 'inspection_completed': False}
print('L1 世界模型:', WORLD_MODEL)
print('L2 业务声明:', DECLARATION)

In [ ]:
def semantic_pipeline(decl, facts, actor_scopes, approved):
    trace = []
    valid = all(o in WORLD_MODEL for o in decl['objects']) and decl['binding'] in GOVERNANCE['bindings'] and decl['effect_type'] in GOVERNANCE['effect_types']
    trace.append(('Describe/Validate', valid))
    if not valid: return trace, 'rejected: 语义声明不合法'
    artifact = {'goal': decl['goal'], 'object_refs': [f'{o}:{facts["lease_id"] if o == "Lease" else facts["space_id"]}' for o in decl['objects']], 'governance': decl['rule']}
    trace.append(('Assemble', True))
    skill_release = {'artifact': artifact, 'effect_policy': 'conditional_write', 'required_scope': 'lease:terminate', 'human_gate': 'mandatory'}
    trace.append(('Materialize', True))
    guard = facts['inspection_completed'] and skill_release['required_scope'] in actor_scopes and approved
    trace.append(('Execute', guard))
    if guard:
        return trace, f"executed: {facts['space_id']} 状态 -> 空置；事实仅回流 L1"
    reasons = []
    if not facts['inspection_completed']: reasons.append('Inspection 未完成')
    if skill_release['required_scope'] not in actor_scopes: reasons.append('scope 不足')
    if not approved: reasons.append('未完成人审')
    return trace, 'blocked: ' + '；'.join(reasons)

trace1, result1 = semantic_pipeline(DECLARATION, RUNTIME_FACTS, {'lease:terminate'}, False)
print('第一次:', result1, trace1)
RUNTIME_FACTS['inspection_completed'] = True
trace2, result2 = semantic_pipeline(DECLARATION, RUNTIME_FACTS, {'lease:terminate'}, True)
print('第二次:', result2, trace2)

In [ ]:
stages = [name for name, _ in trace2]
first = [int(ok) for _, ok in trace1]
second = [int(ok) for _, ok in trace2]
x = np.arange(len(stages))
fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(x - .18, first, .36, label='首次：被 guard 拦截', color='#e76f51')
ax.bar(x + .18, second, .36, label='补齐事实与人审后', color='#2a9d8f')
ax.set_xticks(x, stages); ax.set_ylim(0, 1.25); ax.set_ylabel('阶段通过'); ax.set_title('四层集成的端到端门控')
ax.legend(); ax.grid(axis='y', alpha=.25)
plt.tight_layout(); plt.show()